# 02 Cleaning

Use this notebook to clean the raw data, document your transformations, and export a processed file to `data/processed/`.

# Notebook 02 - Data Cleaning

## Objective

Clean missing values, standardize columns, convert datatypes, engineer features, and export processed dataset.

## Missing Value Handling
## Data Type Conversion
## Feature Engineering
## Export Clean Dataset

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
PROJECT_ROOT = Path('/Users/satyaprakash/nst-dva-capstone-2-project-template')
RAW_PATH = PROJECT_ROOT / 'data/raw/Walmart-Retail-Dataset.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

df = pd.read_csv(RAW_PATH, on_bad_lines='skip', low_memory=False)
df.head()
df.isnull().sum()

city                       7
customer_age               0
customer_name             14
customer_segment        2539
discount                   0
order_date                 0
order_id                   0
order_priority          2112
order_quantity             0
product_base_margin        0
product_category           0
product_container       4734
product_name            4745
product_sub_category    4734
profit                  4734
region                  4734
sales                   4734
ship_date               4734
ship_mode               8157
shipping_cost           4734
state                   4734
unit_price              4734
zip_code                4734
dtype: int64

## Missing Value Handling

In [6]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')


In [7]:
df = df.replace(['\\N', ' ', '', 'NA', 'N/A', 'null'], np.nan)


In [8]:
df = df.drop_duplicates()

## Data Type Conversion

In [9]:
num_cols = [
    'customer_age', 'discount', 'order_quantity',
    'product_base_margin', 'profit', 'sales',
    'shipping_cost', 'unit_price', 'zip_code'
]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

## Feature Engineering

In [10]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

/var/folders/24/bdgz1hbs05b5qhb7yn_9zch00000gn/T/ipykernel_91696/2841956888.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


In [11]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [12]:
df = df[(df['customer_age'] >= 10) & (df['customer_age'] <= 100)]

In [13]:
for col in ['sales', 'profit', 'unit_price', 'shipping_cost']:
    df = df[df[col] >= 0]

In [14]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['ship_date'] = pd.to_datetime(df['ship_date'], errors='coerce')

In [15]:
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

## Export Clean Dataset

In [16]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved cleaned dataset to {PROCESSED_PATH}')

Saved cleaned dataset to /Users/satyaprakash/nst-dva-capstone-2-project-template/data/processed/cleaned_dataset.csv


In [17]:
print(df.isnull().sum())


city                    0
customer_age            0
customer_name           0
customer_segment        0
discount                0
order_date              0
order_id                0
order_priority          0
order_quantity          0
product_base_margin     0
product_category        0
product_container       0
product_name            0
product_sub_category    0
profit                  0
region                  0
sales                   0
ship_date               0
ship_mode               0
shipping_cost           0
state                   0
unit_price              0
zip_code                0
order_year              0
order_month             0
shipping_days           0
dtype: int64


In [18]:
df.head()

,city,customer_age,customer_name,customer_segment,discount,order_date,order_id,order_priority,order_quantity,product_base_margin,...,sales,ship_date,ship_mode,shipping_cost,state,unit_price,zip_code,order_year,order_month,shipping_days
0,Stevens Point,60.0,Dennis Bolton,Corporate,0.17,2020-02-29,a42c8cff-5757-4e94-80b0-807538fefd25,Not Specified,7.0,0.55,...,21.84,2020-03-02,Delivery Truck,3.772509,Wisconsin,3.29,54481.0,2020,2,2
1,Stevens Point,60.0,Dennis Bolton,Corporate,0.17,2020-02-29,1c37f301-564f-40ff-bd7d-73a6c06ede1a,Not Specified,7.0,0.55,...,1811.67,2020-03-07,Delivery Truck,816.340894,Wisconsin,258.98,54481.0,2020,2,7
2,Grapevine,49.0,Anthony Garverick,Small Business,0.05,2021-11-11,ec649eae-535d-4154-b3ef-c4405bd59da9,Medium,42.0,0.69,...,6129.06,2021-11-15,Delivery Truck,4530.505983,Texas,145.98,76051.0,2021,11,4
3,Tempe,30.0,Anne McFarland,Consumer,0.05,2020-08-02,efdcbace-5320-4005-95e2-4c94a896dc8c,Not Specified,30.0,0.37,...,198.90,2020-08-08,Regular Air,128.731505,Arizona,6.68,85281.0,2020,8,6
4,Coconut Creek,80.0,Raymond Fair,Home Office,0.14,2021-08-13,8fd6c0f6-9e28-45b5-ba21-a57021ae304d,Low,44.0,0.52,...,1875.28,2021-08-18,Express Air,33.608385,Florida,42.76,33063.0,2021,8,5
